# Replication of Frey & Šešelja (2020)

**Reference:** Frey, D., & Šešelja, D. (2020). What is the epistemic function of highly idealized agent-based models of scientific inquiry? *Philosophy of the Social Sciences*, 50(4), 347–365.

**Main claim:** Extensions of the basic Zollman model — dynamic objective probabilities (BS), critical interaction (CI), rational inertia (IN), and theory threshold (TT) — affect the speed of convergence on the correct hypothesis, with different effects for complete and cycle networks.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure repo root is in the Python path
sys.path.insert(0, os.path.abspath('..'))

import mesa
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.epistemic_abm.model import Bandit

## Model parameters

### Common parameters (all models)

| Parameter | This model | Original paper | Notes |
|---|---|---|---|
| n | range(4, 11, 2) | identical | |
| a_objective | 0.5 | identical | |
| b_objective | 0.499 | identical | |
| max_priors | 4 | identical | |
| graph | complete, cycle | identical | |
| step_pulls | 1000 | identical | |
| iterations | 1000 | 10000 | We reduced the number of iteration since 1000 is already statistically sufficient |
| max_steps | variable | 100000 | Since we have data on the average time of convergence, we adapted the number of max_steps to reduce compute time|

### Model-specific parameters

| Model | dynamic | criticism | inertia | theory_threshold |
|---|---|---|---|---|
| BS (Basic Setup) | 100 | False | 0 | 0 |
| CI (+ Critical Interaction) | 100 | True | 0 | 0 |
| IN (+ Inertia) | 100 | False | 10 | 0 |
| CI+IN | 100 | True | 10 | 0 |
| TT (+ Theory Threshold) | 100 | False | 0 | 0.1 |

**Deviations from original:**
- `dynamic`: the original model activates objective update every 100 rounds via a boolean flag. In this implementation, `dynamic=100` is equivalent (integer threshold for the update counter).
- `theory_threshold`: the original uses a boolean that maps to a fixed value of 0.1. Here we use `theory_threshold=0.1` directly.

## Common parameters

In [ ]:
common = {
    "n": list(range(4, 11, 2)),
    "a_objective": 0.5,
    "b_objective": 0.499,
    "max_priors": 4,
    "graph": ["complete", "cycle"],
    "step_pulls": 1000,
    "seed": None
}

## Model BS — Basic Setup (dynamic objectives only)

Dynamic shift of objective probabilities towards 1 (for A) and 0 (for B), updated every 100 rounds. How many rounds does each network type need to reach consensus on the correct hypothesis?

In [ ]:
parameters_BS = {
    **common,
    "dynamic": 100,
    "criticism": False,
    "inertia": 0,
    "theory_threshold": 0
}

results_BS = mesa.batch_run(
    Bandit,
    parameters=parameters_BS,
    iterations=1000,
    max_steps=8000
)

df_BS = pd.DataFrame(results_BS)
Path("results/Frey_Seselja_replication/").mkdir(parents=True, exist_ok=True)
df_BS.to_csv("results/Frey_Seselja_replication/df_BS.csv")

In [ ]:
df_BS = pd.read_csv("results/Frey_Seselja_replication/df_BS.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_BS, x="n", y="Convergence Round",
             hue="graph", ax=ax)
ax.set_title("BS — Convergence speed by network type")
ax.set_xlabel("Number of agents (n)")
ax.set_ylabel("Rounds to convergence")
ax.legend(title="Graph type")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/01_basic_setup")
plt.show()

**Result:** [Da compilare dopo l'esecuzione]

## Model CI — Basic Setup + Critical Interaction

Critical interaction: agents slightly modify their own objective probability in response to convincing evidence from neighbors pursuing the competing theory.

In [ ]:
parameters_CI = {
    **common,
    "dynamic": 100,
    "criticism": True,
    "inertia": 0,
    "theory_threshold": 0
}

results_CI = mesa.batch_run(
    Bandit,
    parameters=parameters_CI,
    iterations=1000,
    max_steps=8000
)

df_CI = pd.DataFrame(results_CI)
Path("results/Frey_Seselja_replication/").mkdir(parents=True, exist_ok=True)
df_CI.to_csv("results/Frey_Seselja_replication/df_CI.csv")

In [ ]:
df_CI = pd.read_csv("results/Frey_Seselja_replication/df_CI.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_CI, x="n", y="Convergence Round",
             hue="graph", ax=ax)
ax.set_title("CI — Convergence speed by network type")
ax.set_xlabel("Number of agents (n)")
ax.set_ylabel("Rounds to convergence")
ax.legend(title="Graph type")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/02_critical_interaction")
plt.show()

**Result:** [Da compilare dopo l'esecuzione]

## Model IN — Basic Setup + Rational Inertia

Rational inertia: agents require 10 consecutive rounds of counter-evidence before switching theories.

In [ ]:
parameters_IN = {
    **common,
    "dynamic": 100,
    "criticism": False,
    "inertia": 10,
    "theory_threshold": 0
}

results_IN = mesa.batch_run(
    Bandit,
    parameters=parameters_IN,
    iterations=1000,
    max_steps=5000
)

df_IN = pd.DataFrame(results_IN)
Path("results/Frey_Seselja_replication/").mkdir(parents=True, exist_ok=True)
df_IN.to_csv("results/Frey_Seselja_replication/df_IN.csv")

In [ ]:
df_IN = pd.read_csv("results/Frey_Seselja_replication/df_IN.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_IN, x="n", y="Convergence Round",
             hue="graph", ax=ax)
ax.set_title("IN — Convergence speed by network type")
ax.set_xlabel("Number of agents (n)")
ax.set_ylabel("Rounds to convergence")
ax.legend(title="Graph type")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/03_rational_intertia")
plt.show()

**Result:** [Da compilare dopo l'esecuzione]

## Model CI+IN — Critical Interaction + Rational Inertia

Both extensions combined with the basic setup.

In [ ]:
parameters_CI_IN = {
    **common,
    "dynamic": 100,
    "criticism": True,
    "inertia": 10,
    "theory_threshold": 0
}

results_CI_IN = mesa.batch_run(
    Bandit,
    parameters=parameters_CI_IN,
    iterations=1000,
    max_steps=3000
)

df_CI_IN = pd.DataFrame(results_CI_IN)
Path("results/Frey_Seselja_replication/").mkdir(parents=True, exist_ok=True)
df_CI_IN.to_csv("results/Frey_Seselja_replication/df_CI_IN.csv")

In [ ]:
df_CI_IN = pd.read_csv("results/Frey_Seselja_replication/df_CI_IN.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_CI_IN, x="n", y="Convergence Round",
             hue="graph", ax=ax)
ax.set_title("CI+IN — Convergence speed by network type")
ax.set_xlabel("Number of agents (n)")
ax.set_ylabel("Rounds to convergence")
ax.legend(title="Graph type")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/04_critical_interaction_rational_inertia")
plt.show()

**Result:** [Da compilare dopo l'esecuzione]

## Model TT — Basic Setup + Theory Threshold

Theory threshold: agents require their current theory to be outperformed by at least 0.1 in expected success rate before switching.

In [ ]:
parameters_TT = {
    **common,
    "dynamic": 100,
    "criticism": False,
    "inertia": 0,
    "theory_threshold": 0.1
}

results_TT = mesa.batch_run(
    Bandit,
    parameters=parameters_TT,
    iterations=1000,
    max_steps=50000
)

df_TT = pd.DataFrame(results_TT)
Path("results/Frey_Seselja_replication/").mkdir(parents=True, exist_ok=True)
df_TT.to_csv("results/Frey_Seselja_replication/df_TT.csv")

In [ ]:
df_TT = pd.read_csv("results/Frey_Seselja_replication/df_TT.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_TT, x="n", y="Convergence Round",
             hue="graph", ax=ax)
ax.set_title("TT — Convergence speed by network type")
ax.set_xlabel("Number of agents (n)")
ax.set_ylabel("Rounds to convergence")
ax.legend(title="Graph type")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/05_theory_threshold")
plt.show()

**Result:** [Da compilare dopo l'esecuzione]

## Model CI + TT - Critical interaction + Theory threshold

This model combine the theory threshold of 0.1 with the critical interaction between agents

In [ ]:
parameters_CI_TT = {
    **common,
    "dynamic": 100,
    "criticism": False,
    "inertia": 0,
    "theory_threshold": 0.1
}

results_CI_TT = mesa.batch_run(
    Bandit,
    parameters=parameters_TT,
    iterations=1000,
    max_steps=8000
)

df_CI_TT = pd.DataFrame(results_CI_TT)
Path("results/Frey_Seselja_replication/").mkdir(parents=True, exist_ok=True)
df_CI_TT.to_csv("results/Frey_Seselja_replication/df_CI_TT.csv")

In [ ]:
df_CI_TT = pd.read_csv("results/Frey_Seselja_replication/df_CI_TT.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_CI_TT, x="n", y="Convergence Round",
             hue="graph", ax=ax)
ax.set_title("CI + TT — Convergence speed by network type")
ax.set_xlabel("Number of agents (n)")
ax.set_ylabel("Rounds to convergence")
ax.legend(title="Graph type")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/06_critical_interaction_theory_threshold")
plt.show()

**Results**:

## Comparative overview

Comparison of convergence speed across all model variants.

In [ ]:
######## Takes df from /results if they are not stored in the notebook #########

df_list = [df_BS, df_CI, df_IN, df_CI_IN, df_TT, df_CI_TT]
for df in df_list:
    if not df:
        df = pd.read_csv(f"results/Frey_Seselja_replication/{df}.csv")
df_BS["model"] = "BS"
df_CI["model"] = "CI"
df_IN["model"] = "IN"
df_CI_IN["model"] = "CI+IN"
df_TT["model"] = "TT"
df_CI_TT["model"] = "CI_TT"

df_all = pd.concat([df_BS, df_CI, df_IN, df_CI_IN, df_TT, df_CI_TT])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, graph_type in zip(axes, ["complete", "cycle"]):
    subset = df_all[df_all["graph"] == graph_type]
    sns.lineplot(data=subset, x="n", y="Convergence Round",
                 hue="model", ax=ax)
    ax.set_title(f"{graph_type.capitalize()} graph")
    ax.set_xlabel("Number of agents (n)")
    ax.set_ylabel("Rounds to convergence")
    ax.legend(title="Model")

plt.suptitle("Convergence speed across model variants — Frey & Šešelja (2020)")
plt.tight_layout()
plt.savefig("results/Frey_Seselja_replication/07_comparison")
plt.show()

## Discrepancies

| Model | Claim | Expected | Obtained | Likely cause |
|---|---|---|---|---|
| | | | | |

## Conclusions

[complete after running the simulations]

The replication [successfully / partially / does not] reproduce the main findings of Frey & Šešelja (2020). [Summary dei risultati per ciascun modello]